In [28]:
import numpy as np
import sympy as smp
import qiskit 
from qiskit.circuit import QuantumRegister, QuantumCircuit, Parameter
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit.primitives import StatevectorEstimator, StatevectorSampler

import matplotlib.pyplot as plt

from functools import reduce

import pandas as pd

import scipy.optimize as opt

In [ ]:
n_qubits = 10
zeta = 0
eta = 0
J = 0.1

I2 = np.eye(2)
I = np.eye(2**n_qubits)

X = np.array([
    [0, 1], 
    [1, 0]
])
Z = np.array([
    [1, 0],
    [0, -1]
])
H = np.array([
    [1, 1],
    [1, -1]
]) / np.sqrt(2)

def GateJ(n=n_qubits, j=0, gate=X):
    curr = I2 if j != 0 else gate
    # print(curr)
    for qubit in range(1, n):
        # if qubit==0: continue;
        # print(f"qubit: {qubit}")
        
        if qubit == j:
            curr = np.kron(gate, curr)
            # print('kron X')
            continue
        curr = np.kron(I2, curr)
        # print('kron I')

    return curr

GateJ(n_qubits, 7, H).shape == (1024,1024)

True

$A = \frac{1}{\zeta} ( \sum_{j=1}^{n} X_j + J \sum_{j=1}^{n-1} Z_j Z_{j+1} + \eta I)$

$| b \rangle = H^{\otimes n} | 0 \rangle$

In [ ]:
# calculate first two terms of A (no zeta or eta)
Xsum = np.zeros(shape=(2**n_qubits, 2**n_qubits))
Zsum = np.zeros(shape=(2**n_qubits, 2**n_qubits))
for j in range(0, n_qubits):
    # print(j)
    Xsum += GateJ(j=j)

    if j != n_qubits-1:
        Zsum += GateJ(j=j, gate=Z) @ GateJ(j=j+1, gate=Z)

Zsum *= J
M = Xsum + Zsum

eigenvals = np.linalg.eigvals(M)

In [31]:
κ = 60 # condition number for n=10 qubits

η, ζ = smp.symbols("η ζ")

λmin = min(eigenvals)
λmax = max(eigenvals)

A0 = λmin + η - (ζ / κ)
A1 = λmax + η - ζ

soln = smp.solve([A0, A1], [η, ζ])
eta = soln[η]
zeta = soln[ζ]
soln

{ζ: 20.3847680482924, η: 10.3622570912153}

In [32]:
A = (M + eta * I) / zeta
A

array([[0.552483946078490, 0.0490562363835073, 0.0490562363835073, ...,
        0, 0, 0],
       [0.0490562363835073, 0.542672698801788, 0, ..., 0, 0, 0],
       [0.0490562363835073, 0, 0.532861451525087, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0.532861451525087, 0, 0.0490562363835073],
       [0, 0, 0, ..., 0, 0.542672698801788, 0.0490562363835073],
       [0, 0, 0, ..., 0.0490562363835073, 0.0490562363835073,
        0.552483946078490]], shape=(1024, 1024), dtype=object)

To scale from 3->10 qubits, we need 3 layers:
* Hardware-efficient ansatz
* Layered controlled A gates
* controlled U gate ($H^{\otimes n} | 0 \rangle $)

In [ ]:
# ansatz

def variational_form(α: list, n_qubits: int=10, layers: int=4):
    v = QuantumCircuit(n_qubits)

    for i in range(n_qubits):
        v.ry(, i)

_IncompleteInputError: incomplete input (200658128.py, line 6)